<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; text-align: center; margin-bottom: 10px;">
  <div style="font-size: 64px; margin-bottom: 10px;">🎵</div>
  <h1 style="color: #e94560; font-size: 2.2em; margin: 0 0 8px 0; font-family: Georgia, serif;">Investigación de datos</h1>
  <h2 style="color: #a8dadc; font-size: 1.4em; margin: 0 0 20px 0; font-weight: normal;">Caso Chinook · Tienda de música digital</h2>
  <p style="color: #ccc; font-size: 0.95em; max-width: 500px; margin: 0 auto;">SQL + Python · Análisis exploratorio · Visualización</p>
</div>

## ¿Qué es Chinook?

**Chinook** es una base de datos de ejemplo de una tienda de música digital.
Incluye tablas conectadas entre sí como:

- `Customer` (clientes)
- `Invoice` (facturas)
- `InvoiceLine` (detalle de cada factura)
- `Track` (canciones)
- `Genre` (géneros musicales)

🎯 En este cuaderno vais a trabajar como analistas: formular preguntas, consultar datos y justificar conclusiones con evidencia.

---


# 🔍 Investigación: ¿quién mueve más dinero en Chinook?

Ya sabéis hacer consultas en DB Browser. Ahora el reto es conectar la base de datos desde Python y responder **4 preguntas de negocio** usando `pandas` + `sqlite3`.

No hay una solución única. Si tu consulta funciona y el resultado tiene sentido: está bien.

---

**Normas del reto:**
- Una celda por pregunta con tu consulta SQL dentro de `pd.read_sql(...)`
- El resultado visible (que se vea la tabla)
- **Una línea de texto** debajo explicando qué significa lo que ves
- En cada pregunta, haz primero una versión de prueba (por ejemplo con `LIMIT 5`) y luego la versión final

Al cerrar, revisaremos las soluciones **en conjunto en formato fórum** 👇

## Paso 0 · Conexión a la base de datos

Ejecuta esta celda primero. Si ves una tabla con nombres de tablas, estás dentro ✅

In [12]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt




# Ajusta la ruta si tu Chinook.db está en otra carpeta
con = sqlite3.connect("Chinook.db")

# ¿Qué tablas hay disponibles?
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


---

## Pregunta 0 (calentamiento) — ¿Cuántas facturas hay en total?

Antes de empezar las 4 preguntas, valida que todo funciona con una consulta muy corta.

Si esta sale bien, ya tienes conexión y sintaxis controladas ✅

In [13]:
# Pregunta 0: total de facturas
pd.read_sql(
    """
    SELECT COUNT(*) AS total_facturas
    FROM Invoice
    """,
    con
)

,total_facturas
0,412


---

## Pregunta 1 — ¿Qué empleado tiene más clientes asignados?

> 💡 Pista: hay una columna en la tabla `Customer` que apunta a la tabla `Employee`. ¿Cuál es?

Escribe tu consulta en la celda de abajo y muestra el resultado.

Cuando lo tengas, añade una línea en texto explicando qué significa el número que ves.

In [14]:
# Pregunta 1: ¿Qué empleado tiene más clientes asignados?
# Escribe tu consulta aquí:

pd.read_sql("""
    SELECT SupportRepId AS Empleado_ID, COUNT(*) AS Clientes
            FROM Customer 
            GROUP BY Empleado_ID
            ORDER BY Clientes DESC 
            LIMIT 1
""", con)

,Empleado_ID,Clientes
0,3,21


En ejercicio anterior podemos ver que el empleado con ID 3 tiene mas clientes asignados - en total 21.

**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

1. identifica empleados en tabla Customer,
2. cuenta cuantos clientes tiene cada empleado
3. agrupa los clientes por empleado
4. cuenta cuantos clientes hay en cada grupo.
5. ordena de mayor a menor
6. muestra solo el empleado con mas clientes (limita a 1) 

---

## Pregunta 2 — ¿En qué mes del año se factura más? ¿Y menos?

> 💡 Pista: la fecha de factura está guardada como texto. Puedes extraer el mes con `strftime('%m', InvoiceDate)`.

Cuando lo tengas, añade una línea explicando si el patrón tiene sentido o te parece sorprendente.

In [15]:
# Pregunta 2: ¿En qué mes se factura más? ¿Y menos?
# Escribe tu consulta aquí:

pd.read_sql("""
  SELECT strftime('%m', InvoiceDate) AS Mes, SUM(Total) AS Total_Facturas
            FROM Invoice 
            GROUP BY Mes
            ORDER BY Total_Facturas DESC 
            Limit 1
""", con)


pd.read_sql("""
  SELECT strftime('%m', InvoiceDate) AS Mes, SUM(Total) AS Total_Facturas
            FROM Invoice 
            GROUP BY Mes
            ORDER BY Total_Facturas ASC 
            Limit 1
""", con)


,Mes,Total_Facturas
0,11,186.24


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

1. Primero abre la tabla Invoice.
2. Ahora se extrae el mes de la fecha.
3. Ahora agrupa las facturas por mes.
4. Dentro de cada grupo suma el dinero facturado.
5. Ahora ordena de mayor a menor facturación              o de menor a mayor.
6. Finalmente muestra solo la primera fila.

# El mes con mayor facturacion es Enero. El mes con menor facturacion es Noviembre
---

## Pregunta 3 — ¿Hay clientes que hayan comprado solo una vez? ¿Cuántos son?

> 💡 Pista: una compra = una factura en la tabla `Invoice`. Agrupa por cliente y filtra los que solo tienen 1.

¿Sorprende la cantidad? Escribe tu conclusión debajo del resultado.

In [16]:
# Pregunta 3: ¿Cuántos clientes han comprado solo una vez?
# Escribe tu consulta aquí:

pd.read_sql("""
    SELECT CustomerId, COUNT(*) AS Compras
            FROM Invoice 
            GROUP BY CustomerId 
            HAVING Compras = 1 
        

""", con)

,CustomerId,Compras


1. decides qué columnas quieres ver en el resultado.
CustomerId - el id del cliente
COUNT(*)- cuenta cuantas facturas tiene ese cliente
AS Compras - le ponemos el nombre Compras a esa columna
2. lo sacas de tabla Invoice
3. agrupa por clientes
4. cuenta facturas de clientes. COUNT(*) cuenta las filas dentro de cada grupo
5. filtramos grupo. Solo deja los clientes que tienen 1 compra. 

WHERE filtra filas antes de agrupar. HAVING filtra grupos después de GROUP BY
Por eso aquí usamos HAVING, porque estamos filtrando el resultado de COUNT(*). 
COUNT(*) se calcula después del GROUP BY, por eso WHERE no lo puede usar.

La tabla esta vacia que significa que cada cliente hizo la compra mas de 1 vez


---

## Pregunta 4 — ¿Cuál es el género musical más vendido por tracks comprados?

> 💡 Esta la haremos en 2 pasos:
> 1) Une `InvoiceLine` con `Track` para comprobar que puedes llegar a `TrackId` y `GenreId`.
> 2) Añade `Genre`, agrupa y ordena para obtener el top.

Esta es la más difícil. Si llegas aquí, enhorabuena 🎉

In [17]:
# Pregunta 4: ¿Cuál es el género más vendido por número de tracks comprados?

# Paso 1 (prueba): valida el cruce InvoiceLine + Track
pd.read_sql("""

SELECT InvoiceLine.TrackId, Track.GenreId
FROM InvoiceLine
JOIN Track ON InvoiceLine.TrackId = Track.TrackId
LIMIT 10
            
   """ , con)

# Paso 2 (final): añade Genre, agrupa y ordena
pd.read_sql("""
SELECT Genre.Name, COUNT(*) 
FROM InvoiceLine
JOIN Track ON InvoiceLine.TrackId = Track.TrackId
JOIN Genre ON Track.GenreId = Genre.GenreId
GROUP BY Genre.Name
ORDER BY COUNT(*) DESC
""", con)

,Name,COUNT(*)
0,Rock,835
1,Latin,386
2,Metal,264
3,Alternative & Punk,244
4,Jazz,80
5,Blues,61
6,TV Shows,47
7,R&B/Soul,41
8,Classical,41
9,Reggae,30


1. quiero ver ID del track que se compró y el género al que pertenece ese track
2. InvoiceLine contiene los tracks comprados, TrackID - Identyfica la cancion
3. conecta la tabla InvoiceLine con la tabla Track
4. Une las filas donde el TrackId de InvoiceLine sea igual al TrackId de Track 

5. Añdimos la tabla GEnre
6. Contamos ventas por cada genero
7. busca el Track  (cancion) que corresponde a cada TrackId vendido
8. busca el nombre del genero de cada canción
9. agrupa por nombre del genero
10. Ordena de mayor a menor


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

---

## Cierre en fórum (sin entrega)

- No hay entrega individual hoy
- Guarda tus consultas para poder enseñarlas si te toca compartir
- Cerraremos la sesión resolviendo dudas y comparando enfoques **en conjunto**
- Piensa qué pregunta te costó más y por qué, para comentarla en el fórum